In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import re
import string

In [6]:
data_fake = pd.read_csv('Fake.csv')
data_true = pd.read_csv('True.csv')

In [ ]:
data_fake.head()

In [ ]:
data_true.head()

In [9]:
data_fake["class"] = 0
data_true["class"] = 1

In [ ]:
data_fake.shape, data_true.shape

In [11]:
data_fake_manual_testing = data_fake.tail(10)
for i in range(23480, 23470, -1):
    data_fake.drop([i], axis = 0, inplace = True)

data_true_manual_testing = data_true.tail(10)
for i in range(21416, 21406, -1):
    data_true.drop([i], axis = 0, inplace = True)

In [ ]:
data_fake.shape, data_true.shape

In [ ]:
data_fake_manual_testing["class"] = 0
data_true_manual_testing["class"] = 1

In [ ]:
data_fake_manual_testing.head(10)

In [ ]:
data_true_manual_testing.head(10)

In [16]:
data_merge = pd.concat([data_fake, data_true], axis = 0)
data_merge.shape

(44878, 5)

In [ ]:
data_merge.columns

In [18]:
data = data_merge.drop(["title", "subject", "date"], axis = 1)

In [ ]:
data.isnull().sum()

In [ ]:
data = data.sample(frac = 1)
data.head()

In [22]:
data.reset_index(inplace = True)
data.drop(['index'], axis = 1, inplace = True)

In [23]:
data.columns

Index(['text', 'class'], dtype='object')

In [ ]:
data.head()

In [25]:
def wordopt(text):
    text = text. lower()
    text = re.sub('\[ .*? \]', '', text)
    text = re.sub("\\W", " ", text)
    text = re.sub('https ?: //\S+|www\.\S+', '', text)
    text = re.sub(' <.*? >+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

In [26]:
data['text'] = data['text'].apply(wordopt)

In [27]:
x = data['text']
y = data['class']

In [28]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.25)

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
LR = LogisticRegression()
LR.fit(xv_train, y_train)

In [31]:
pred_lr = LR.predict(xv_test)

In [32]:
LR.score(xv_test, y_test)

0.986096256684492

In [ ]:
print(classification_report(y_test, pred_lr))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

DT = DecisionTreeClassifier()
DT.fit(xv_train, y_train)

In [35]:
pred_dt = DT.predict(xv_test)

In [ ]:
DT.score(xv_test, y_test)

In [ ]:
print(classification_report(y_test, pred_dt))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
GB = GradientBoostingClassifier(random_state = 0)
GB.fit(xv_train, y_train)

In [39]:
pred_gb = GB.predict(xv_test)

In [ ]:
GB.score(xv_test, y_test)

In [ ]:
print(classification_report(y_test, pred_gb))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
RF = RandomForestClassifier(random_state = 0)
RF.fit(xv_train, y_train)

In [43]:
pred_rf = RF.predict(xv_test)

In [ ]:
RF.score(xv_test, y_test)

In [ ]:
print(classification_report(y_test, pred_rf))

In [3]:
def output_lable(n):
    if n == 0:
        return "Fake News"
    elif n == 1:
        return "Not A Fake News"
def manual_testing(news):
    testing_news = {"text": [news]}
    new_def_test = pd.DataFrame(testing_news)
    new_def_test["text"] = new_def_test["text"].apply(wordopt)
    new_x_test = new_def_test["text"]
    new_xv_test = vectorization.transform(new_x_test)
    pred_LR = LR.predict(new_xv_test)
    pred_DT = DT.predict(new_xv_test)
    pred_GB = GB.predict(new_xv_test)
    pred_RF = RF.predict(new_xv_test)
    print("\n\nLR Prediction: {} \nDT Prediction: {} \nGB Prediction: {} \nRF Prediction: {}".format(output_lable(pred_LR[0]), output_lable(pred_DT[0]), output_lable(pred_GB[0]), output_lable(pred_RF[0])))

In [ ]:
news = str(input())
manual_testing(news)